# Module 6 Lab 2: Sequential LangChain Workflow for Corporate Operations

## Objective
Build a **four-step sequential chain** with LangChain Expression Language (LCEL) that helps a routine operations team handle everyday requests: office supplies, building badges, equipment faults, vendor invoices and meeting room issues.

## Learning Outcomes
By the end of this lab, you will:
- Build a sequential chain with LCEL (`prompt | llm | StrOutputParser()`)
- Write a prompt template for each step
- Pass each step's output into the next step
- Save the results to a JSON file

## Use Case: Routine Corporate Operations
Routine requests are often handled by hand: someone summarizes the request, works out the cause, judges the impact, then writes an action plan. This chain does those four steps in order.

```
Request (type, requester, details, due date)
    |
[Step 1: Summary]           -> summary, category, priority
    |  (uses step 1)
[Step 2: Cause & blockers]  -> likely cause, blockers
    |  (uses steps 1-2)
[Step 3: Business impact]   -> who is affected, impact if late
    |  (uses steps 1-3)
[Step 4: Action plan]       -> next steps with owners and time targets
```

The sample data is fictional. Do not paste real employee, customer or financial records into the prompts.

## Part 1: Setup
Put your `OPENAI_API_KEY` in a `.env` file in this folder. Install the packages with `pip install -r requirements_module6_lab2.txt` (no changes to the package versions).

In [ ]:
import os
import json

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Load OPENAI_API_KEY from the .env file in this folder
load_dotenv()

print('OpenAI API key configured:', 'Yes' if os.getenv('OPENAI_API_KEY') else 'No')

## Part 2: Load the Requests
The file `data/ops_requests.json` has five routine operations requests.

In [ ]:
with open('data/ops_requests.json', 'r', encoding='utf-8') as f:
    requests = json.load(f)['requests']

print(f'Loaded {len(requests)} requests')
for r in requests:
    print(r['request_id'], '-', r['request_type'], '-', r['department'], '- due', r['due_date'])

## Part 3: Prompt Templates
Each step has its own prompt. Each prompt uses the output of the steps before it. The answer format is written into the prompt, so the output is easy to read and easy to pass on.

In [ ]:
# Step 1: summary of the request
summary_prompt = PromptTemplate.from_template('''You are an operations coordinator at a corporate office.
Read the routine operations request below.

Request type: {request_type}
Requester: {requester} ({department})
Details: {details}
Needed by: {due_date}

Answer in this format:
Summary: <two sentences describing the request>
Category: <one of: Facilities, IT Access, Supplies, Equipment, Vendor, Other>
Priority: <one of: High, Medium, Low>
''')

# Step 2: likely cause and blockers (uses step 1)
cause_prompt = PromptTemplate.from_template('''You are an operations analyst. Using the request and summary below,
explain why the issue happened and what is blocking it from being resolved.

Request type: {request_type}
Details: {details}
Summary: {summary}

Answer in this format:
Likely cause: <one or two sentences>
Blockers:
- <blocker 1>
- <blocker 2>
Confidence: <High, Medium or Low>
''')

# Step 3: business impact (uses steps 1 and 2)
impact_prompt = PromptTemplate.from_template('''You are an operations manager. Using the summary and likely cause below,
describe the business impact of this request if it is not resolved on time.

Summary: {summary}
Likely cause and blockers: {cause}
Needed by: {due_date}

Answer in this format:
Who is affected: <teams, people or customers>
Impact if delayed: <one or two sentences>
Cost level: <High, Medium or Low>
Deadline at risk: <Yes or No>
''')

# Step 4: action plan with owners (uses steps 1, 2 and 3)
action_prompt = PromptTemplate.from_template('''You are a senior operations coordinator. Using the information below,
write a short action plan to resolve this request.

Request type: {request_type}
Summary: {summary}
Likely cause and blockers: {cause}
Business impact: {impact}
Needed by: {due_date}

Answer in this format:
Immediate (next 30 minutes):
1. <action> - Owner: <team>
Same day:
1. <action> - Owner: <team>
Follow-up to prevent recurrence:
1. <action> - Owner: <team>
''')

print('Prompt templates ready: summary, cause, impact, action')

## Part 4: Build the Chains
Each chain is `prompt | llm | StrOutputParser()`. The parser turns the model reply into plain text, so the output of one step can go straight into the next prompt.

In [ ]:
llm = ChatOpenAI(
    model='gpt-4o-mini',  # the lab default; change only if your account needs another model
    temperature=0.3,
    api_key=os.getenv('OPENAI_API_KEY'),
)
parser = StrOutputParser()

summary_chain = summary_prompt | llm | parser
cause_chain = cause_prompt | llm | parser
impact_chain = impact_prompt | llm | parser
action_chain = action_prompt | llm | parser

print('Chains ready: summary, cause, impact, action')

## Part 5: Run the Sequential Chain on One Request
Run the four steps in order on the first request. Each step prints its output before the next step starts.

In [ ]:
request = requests[0]
print('Request:', request['request_id'], '-', request['request_type'])
print()

# Step 1: summary
summary = summary_chain.invoke({
    'request_type': request['request_type'],
    'requester': request['requester'],
    'department': request['department'],
    'details': request['details'],
    'due_date': request['due_date'],
})
print('[Step 1/4] SUMMARY')
print(summary)
print()

# Step 2: cause and blockers (uses step 1)
cause = cause_chain.invoke({
    'request_type': request['request_type'],
    'details': request['details'],
    'summary': summary,
})
print('[Step 2/4] CAUSE AND BLOCKERS')
print(cause)
print()

# Step 3: business impact (uses steps 1 and 2)
impact = impact_chain.invoke({
    'summary': summary,
    'cause': cause,
    'due_date': request['due_date'],
})
print('[Step 3/4] BUSINESS IMPACT')
print(impact)
print()

# Step 4: action plan (uses steps 1, 2 and 3)
action_plan = action_chain.invoke({
    'request_type': request['request_type'],
    'summary': summary,
    'cause': cause,
    'impact': impact,
    'due_date': request['due_date'],
})
print('[Step 4/4] ACTION PLAN')
print(action_plan)

## Part 6: Run All Requests and Save the Results
Run the same four steps on every request. Each result is a plain dictionary, and the list is saved as JSON.

In [ ]:
results = []

for request in requests:
    print('Processing', request['request_id'], '-', request['request_type'])

    summary = summary_chain.invoke({
        'request_type': request['request_type'],
        'requester': request['requester'],
        'department': request['department'],
        'details': request['details'],
        'due_date': request['due_date'],
    })
    cause = cause_chain.invoke({
        'request_type': request['request_type'],
        'details': request['details'],
        'summary': summary,
    })
    impact = impact_chain.invoke({
        'summary': summary,
        'cause': cause,
        'due_date': request['due_date'],
    })
    action_plan = action_chain.invoke({
        'request_type': request['request_type'],
        'summary': summary,
        'cause': cause,
        'impact': impact,
        'due_date': request['due_date'],
    })

    results.append({
        'request_id': request['request_id'],
        'request_type': request['request_type'],
        'summary': summary,
        'cause': cause,
        'impact': impact,
        'action_plan': action_plan,
    })

with open('ops_analysis_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print('Saved', len(results), 'results to ops_analysis_results.json')

## Part 7: Read the Report
Print each saved result as a short report for the operations team.

In [ ]:
for r in results:
    print('=' * 70)
    print(r['request_id'], '-', r['request_type'])
    print('=' * 70)
    print('1. SUMMARY')
    print(r['summary'])
    print()
    print('2. CAUSE AND BLOCKERS')
    print(r['cause'])
    print()
    print('3. BUSINESS IMPACT')
    print(r['impact'])
    print()
    print('4. ACTION PLAN')
    print(r['action_plan'])
    print()

## Key Concepts
- `prompt | llm | StrOutputParser()` turns a prompt into plain text.
- Each step's output is a string, so it goes straight into the next step's prompt.
- The workflow uses plain functions, dictionaries and JSON only. There are no classes and no Pydantic models.

## Exercises
1. Add a fifth step that lists the checks needed to confirm the request is closed.
2. Add a new request type that fits your team, for example a room booking or a parking pass.
3. Compare the outputs at `temperature=0` and `temperature=0.7`.
4. Change the category list in the summary prompt to match your team's own categories.